# MASLD-Bench · Colab bridge worker

Unattended GPU worker for the MASLD-Bench platform. Talks to the laptop
platform **through Google Drive** (no API keys, no tunnels):

1. The platform writes job specs to `MyDrive/masld-bridge/outbox/*.json`.
2. This notebook (scheduled with **Colab Pro+ → Schedule**, e.g. hourly)
   claims queued `eval_local` jobs and runs them on the Colab GPU.
3. Result archives land in `MyDrive/masld-bridge/inbox/`, where the
   platform supervisor ingests + scores them automatically.

**Setup (one time):** `Runtime → Change runtime type → GPU (T4/L4)`, then
use the **Schedule** (clock) icon to run this notebook at your cadence.
Every run is idempotent: it processes only `queued` jobs and exits.

In [1]:
#@title 1 · Mount Drive + configure the bridge
import glob, gzip, json, time
from pathlib import Path

BRIDGE_DIR = "/content/drive/MyDrive/masld-bridge"  #@param
# Optional: research repo synced to Drive (default corpus location).
RESEARCH_DIR = "/content/drive/MyDrive/masld-llm"  #@param

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    raise SystemExit("This notebook must run on Google Colab.")

for sub in ("outbox", "inbox", "done"):
    Path(BRIDGE_DIR, sub).mkdir(parents=True, exist_ok=True)
print("bridge ready:", BRIDGE_DIR)

Mounted at /content/drive
bridge ready: /content/drive/MyDrive/masld-bridge


In [2]:
#@title 2 · Dependencies — remove torchao (proven root cause), keep image stack
import subprocess, sys, importlib.metadata as md

# peft's dispatch_torchao only skips its (broken) torchao import when torchao
# is ABSENT. Plain LoRA adapters need no torchao, and the bf16 load path in
# transformers never touches it. Uninstall; if the image blocks that,
# downgrade to 0.7.0 (the last version carrying LinearActivationQuantizedTensor).
try:
    subprocess.run([sys.executable, "-m", "pip", "-q", "uninstall", "-y", "torchao"],
                   check=True)
    print("torchao: removed (peft dispatch inert — intended)")
except subprocess.CalledProcessError:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "torchao==0.7.0"],
                   check=True)
    print("torchao: downgraded to 0.7.0 (has LinearActivationQuantizedTensor)")

# No -U: keep the image's peft/transformers (the -U variants pulled the broken
# combos). This line only fills anything genuinely missing.
subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                "transformers", "peft", "accelerate", "bitsandbytes"], check=True)

print("transformers", md.version("transformers"))
print("peft", md.version("peft"))
print("accelerate", md.version("accelerate"))
print("deps ok")



torchao: removed (peft dispatch inert — intended)
transformers 5.20.2
peft 0.21.2
accelerate 1.15.0
deps ok


In [3]:
#@title 3 · Claim the next queued job
def read_json(p):
    return json.loads(Path(p).read_text())

def write_json(p, obj):
    Path(p).write_text(json.dumps(obj, indent=2))

def claim_next_job():
    for spec_path in sorted(glob.glob(f"{BRIDGE_DIR}/outbox/*.json")):
        job = read_json(spec_path)
        if job.get("status") != "queued":
            continue
        job["status"] = "running"
        job["updated_at"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
        write_json(spec_path, job)
        write_json(f"{BRIDGE_DIR}/inbox/{job['job_id']}.status.json", job)
        return job
    return None

JOB = claim_next_job()
print("claimed:", json.dumps(JOB, indent=2)[:800] if JOB else "(no queued jobs — nothing to do)")

claimed: {
  "job_id": "89xx8b55044c",
  "type": "eval_local",
  "status": "running",
  "created_at": "2026-10-30T13:02:34+00:00",
  "updated_at": "2026-10-30T13:04:41Z",
  "requested_by": "zcode-operator",
  "params": {
    "base_model": "Qwen/Qwen3-8B",
    "adapter_path": "/content/drive/MyDrive/masld-llm/checkpoints/qwen3-8b-masld-sft-v7",
    "dataset_tag": "MB-89xx",
    "batch": 8,
    "max_new_tokens": 64,
    "max_length": 4096,
    "items": [
      {
        "task": "MB-8901",
        "split": "test",
        "sample": "RNA015584",
        "source": "S-BSST1614",
        "prompt": "Below is a bulk RNA-seq profile of plasma cell-free RNA.\nPatient: plasma cfRNA profile.\nGene expression (cfRNA): ENSG00000182389=-0.99, ENSG00000161249=-0.97, ENSG00000239665=-1.01, ENSG00000089169=-1.20, ENS


In [4]:
#@title 4 · Load corpus items for the job
def load_records(job):
    params = job.get("params", {})
    if params.get("items"):  # records embedded in the job spec
        return params["items"]
    corpus = params.get("corpus_drive_path") or Path(
        RESEARCH_DIR, "prompts", params.get("corpus", "MB-20xx_liver_test.jsonl.gz")
    )
    records = []
    with gzip.open(corpus, "rt") as f:
        for line in f:
            obj = json.loads(line)
            if "__meta__" in obj:
                continue
            records.append(obj)
    return records

if JOB:
    RECORDS = load_records(JOB)
    print(f"{len(RECORDS)} records; tasks:", sorted({r['task'] for r in RECORDS}))

873 records; tasks: ['MB-8901', 'MB-8902', 'MB-8905', 'MB-8911', 'MB-8912', 'MB-8915']


In [5]:
#@title 5 · Run the evaluation (greedy, newline-stopped, batched)
import torch
import gzip
import json
from pathlib import Path
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

def finish(job, status, error=None):
    job["status"] = status
    job["updated_at"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    if error:
        job["error"] = str(error)[:500]
    write_json(f"{BRIDGE_DIR}/inbox/{job['job_id']}.status.json", job)
    spec = f"{BRIDGE_DIR}/outbox/{job['job_id']}.json"
    if Path(spec).exists():
        write_json(spec, job)

if JOB:
    try:
        # Reset status if it failed previously
        if JOB.get("status") == "failed":
            JOB["status"] = "running"
            JOB["error"] = None

        params = JOB["params"]
        device = "cuda" if torch.cuda.is_available() else "cpu"
        base_model = params.get("base_model", "Qwen/Qwen3-0.6B")
        adapter = params.get("adapter_path")  # Drive path to the PEFT adapter
        assert adapter, "job params must include adapter_path (Drive path)"
        tok = AutoTokenizer.from_pretrained(base_model, padding_side="left")
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        # Use torch_dtype instead of dtype
        model = AutoModelForCausalLM.from_pretrained(base_model, torch_dtype=torch.bfloat16)
        model = PeftModel.from_pretrained(model, adapter).to(device).eval()

        batch = int(params.get("batch", 8))
        max_new = int(params.get("max_new_tokens", 24))
        out_records = []
        for i in range(0, len(RECORDS), batch):
            chunk = RECORDS[i:i + batch]
            enc = tok([r["prompt"] for r in chunk], padding=True, truncation=True,
                      max_length=int(params.get("max_length", 4096)), return_tensors="pt").to(device)
            with torch.no_grad():
                gen = model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                                     pad_token_id=tok.pad_token_id)
            for j, r in enumerate(chunk):
                text = tok.decode(gen[j][enc["input_ids"].shape[1]:], skip_special_tokens=True)
                out_records.append({
                    "task": r["task"], "sample": r["sample"], "answer": r.get("answer"),
                    "raw": text.split("\n")[0].strip(), "error": None,
                    "model_version": Path(adapter).name, "parsed": None,
                })
            if (i // batch) % 10 == 0:
                print(f"  {i + len(chunk)}/{len(RECORDS)}")

        model_tag = Path(adapter).name.replace("/", "__")
        dataset_tag = params.get("dataset_tag", "MB-20xx")
        archive = Path(BRIDGE_DIR, "inbox",
                       f"{JOB['job_id']}__{model_tag}__{dataset_tag}.jsonl.gz")
        with gzip.open(archive, "wt") as f:
            for rec in out_records:
                f.write(json.dumps(rec) + "\n")
        print(f"archive -> {archive} ({len(out_records)} records)")
        finish(JOB, "done")
    except Exception as exc:  # report failure through the bus
        print("JOB FAILED:", exc)
        finish(JOB, "failed", exc)
        raise
else:
    print("nothing to do")



config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (3 outputs left out)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

  8/873
  88/873
  168/873
  248/873
  328/873
  408/873
  488/873
  568/873
  648/873
  728/873
  808/873
archive -> /content/drive/MyDrive/masld-bridge/inbox/89xx8b55044c__qwen3-8b-masld-sft-v7__MB-89xx.jsonl.gz (873 records)


## Dispatching jobs from the laptop

```bash
masld bridge-dispatch eval_local '{
  "base_model": "Qwen/Qwen3-0.6B",
  "adapter_path": "/content/drive/MyDrive/masld-llm/checkpoints/qwen3-0.6b-masld-sft",
  "corpus": "MB-20xx_liver_test.jsonl.gz",
  "dataset_tag": "MB-20xx"
}'
```

`masld agent` (or `masld bridge-sync`) picks the finished archive out of
the inbox, registers and scores it, and appends an audit event.
Job specs and status files are plain JSON — inspect
`MyDrive/masld-bridge/{outbox,inbox,done}` at any time.